# Attention probabilities and rollout

Inspect a seeded untrained model. The maps illustrate the API; their structure is not evidence of learned semantic attention.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / "edgeformer").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "edgeformer").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
torch.set_num_threads(1)
torch.manual_seed(42)
from edgeformer.models import TinyViT

model = TinyViT(
    image_size=16,
    patch_size=4,
    num_classes=4,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
)

In [ ]:
model.eval()
image = torch.randn(3, 16, 16)
with torch.inference_mode():
    logits, maps = model(image.unsqueeze(0), return_attention=True)
print("Logits:", logits.shape)
print("Maps:", [tuple(m.shape) for m in maps])
for m in maps:
    torch.testing.assert_close(m.sum(-1), torch.ones_like(m.sum(-1)))

In [ ]:
from edgeformer.visualization.attention_map import save_attention_report

output = ROOT / "runs" / "notebook_attention.png"
save_attention_report(model, image, output)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.imshow(plt.imread(output))
plt.axis("off")

`return_attention=True` uses explicit attention and returns probabilities before dropout. The normal forward uses SDPA unless configured otherwise. Rollout averages heads and incorporates residual mixing; it is an inspection heuristic.